# Ingest KO + EC annotations → BERDL lakehouse (`nmdc_results` namespace)

Loads the two Parquets produced by `fetch_ko_ec_annotations.ipynb` into the
`nmdc` tenant's `nmdc_results` namespace as Delta tables — alongside the
taxonomy summary tables already there.

On-pod, so no SSH tunnels / proxy. Same two-phase flow as
`ingest_taxonomy_summaries.ipynb`:
1. **Upload** local Parquets to MinIO bronze under
   `tenant-general-warehouse/nmdc/datasets/results/`.
2. **Ingest** writes Delta tables under
   `s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_results.db/`.

After ingest, query as e.g.
`SELECT * FROM nmdc_results.annotation_kegg_orthology WHERE annotation_id = 'KO:K00835' LIMIT 10`.

### Configuration

In [ ]:
import os
from pathlib import Path

TENANT      = "nmdc"
DATASET     = "results"
BUCKET      = "cdm-lake"
MODE        = "overwrite"   # "overwrite" or "append"

SOURCE_DIR  = Path(os.environ.get("KO_EC_OUT_DIR", "loaded_ko_ec"))

NAMESPACE     = f"{TENANT}_{DATASET}"
BRONZE_PREFIX = f"tenant-general-warehouse/{TENANT}/datasets/{DATASET}"
BRONZE_BASE   = f"s3a://{BUCKET}/{BRONZE_PREFIX}"
SILVER_BASE   = f"s3a://{BUCKET}/tenant-sql-warehouse/{TENANT}/{NAMESPACE}.db"

print(f"Tenant       : {TENANT}")
print(f"Dataset      : {DATASET}")
print(f"Namespace    : {NAMESPACE}")
print(f"Mode         : {MODE}")
print(f"Source dir   : {SOURCE_DIR.resolve()}")
print(f"Bronze base  : {BRONZE_BASE}")
print(f"Silver base  : {SILVER_BASE}")

### Setup — Spark Connect + MinIO

Pass `tenant_name=TENANT` so Spark writes to the nmdc warehouse, not the
user's personal one.

In [ ]:
spark = get_spark_session(app_name=f"ingest_ko_ec_into_{NAMESPACE}", tenant_name=TENANT)
mc    = get_minio_client()
print(f"Spark version: {spark.version}")
print(f"MinIO endpoint: {mc._base_url._url.netloc}")

### Inputs — locate source Parquets

In [ ]:
EXPECTED_TABLES = [
    "annotation_kegg_orthology",
    "annotation_enzyme_commission",
]

plan = []
for name in EXPECTED_TABLES:
    local = SOURCE_DIR / f"{name}.parquet"
    if not local.exists():
        print(f"  -- {name}: missing on disk, will skip")
        continue
    size_mb = local.stat().st_size / 1024 / 1024
    plan.append({"name": name, "local": local, "size_mb": size_mb})
    print(f"  OK {name}: {size_mb:,.1f} MB")

if not plan:
    raise RuntimeError(f"No source Parquets found under {SOURCE_DIR.resolve()}")
print(f"\n{len(plan)} table(s) will be ingested.")

### Phase 1 — upload to MinIO bronze

Spark executors run on other cluster nodes and can't see the notebook's local
filesystem, so we upload via the **driver-side MinIO client**. Column-name
sanitization is included for safety, even though BLAST schema columns are all
Delta-safe today.

In [ ]:
import re
import time
import pyarrow.parquet as pq

# Delta forbids column names containing spaces, commas, semicolons, braces,
# parens, newline, tab, or equals. Replace with underscores defensively.
_ILLEGAL_COL_CHARS_RE = re.compile(r"[ ,;{}()\n\t=]+")


def _sanitize_col(name: str) -> str:
    return _ILLEGAL_COL_CHARS_RE.sub("_", name).strip("_")


def _sanitize_parquet(local_path: Path) -> Path:
    table     = pq.read_table(local_path)
    original  = table.column_names
    sanitized = [_sanitize_col(c) for c in original]
    if original == sanitized:
        return local_path
    renamed = [(o, s) for o, s in zip(original, sanitized) if o != s]
    table = table.rename_columns(sanitized)
    fixed = local_path.with_suffix(".sanitized.parquet")
    pq.write_table(table, fixed)
    print(f"    sanitized {len(renamed)} column name(s): {renamed}")
    return fixed


for entry in plan:
    name        = entry["name"]
    local_path  = entry["local"].resolve()
    upload_path = _sanitize_parquet(local_path)
    object_key  = f"{BRONZE_PREFIX}/{name}.parquet"
    bronze_uri  = f"s3a://{BUCKET}/{object_key}"
    print(f"  uploading {name} ({entry['size_mb']:,.1f} MB) → {bronze_uri}")
    t0 = time.monotonic()
    mc.fput_object(BUCKET, object_key, str(upload_path))
    print(f"    done in {time.monotonic() - t0:.1f}s")
    entry["bronze"]     = bronze_uri
    entry["object_key"] = object_key

print("\nAll uploads complete.")

### Phase 2 — build the ingest config

In [ ]:
config = {
    "tenant":  TENANT,
    "dataset": DATASET,
    "paths": {
        "bronze_base": BRONZE_BASE,
        "silver_base": SILVER_BASE,
    },
    "defaults": {
        "parquet": {},
    },
    "tables": [
        {
            "name":         entry["name"],
            "format":       "parquet",
            "bronze_path":  f"{entry['name']}.parquet",
            "write_mode":   MODE,
        }
        for entry in plan
    ],
}

import json
print(json.dumps(config, indent=2))

### Phase 3 — run ingest

In [ ]:
report = ingest(config=config, spark=spark, minio_client=mc)

import json
print(json.dumps(report, indent=2, default=str))

### Verify

In [ ]:
import pyarrow.parquet as pq

print(f"=== tables in namespace `{NAMESPACE}` ===")
spark.sql(f"SHOW TABLES IN {NAMESPACE}").show(truncate=False)

print(f"\n=== row counts: lakehouse vs source (KO/EC tables) ===")
for entry in plan:
    name        = entry["name"]
    delta_count = spark.sql(f"SELECT COUNT(*) AS n FROM {NAMESPACE}.{name}").collect()[0]["n"]
    src_count   = pq.read_metadata(str(entry["local"])).num_rows
    match       = "OK" if delta_count == src_count else "MISMATCH"
    print(f"  {match:9s} {name}: delta={delta_count:>14,}  source={src_count:>14,}")

In [ ]:
# Spot-check: schema + sample rows
spark.sql(f"DESCRIBE {NAMESPACE}.annotation_kegg_orthology").show(truncate=False)
spark.sql(f"SELECT * FROM {NAMESPACE}.annotation_kegg_orthology LIMIT 5").show(truncate=False)